## HW8 — пропуски и выбросы

Заполните пропуски и исследуйте выбросы в подготовленных данных Taxi. Порядок исходных ячеек сохранён; полный CSV не пересчитывался.

In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv('processed_data_with_new_features.csv', index_col='id')

In [ ]:
data.head()

### Пропуски

В passenger_count и distance_km есть пропущенные значения. Посчитайте их и проверьте остальные признаки.

In [ ]:
### Выведите количество пропусков в данных колонках
### На всякий случай, проверьте целостность данных в остальных
passanger_count_miss = data['passenger_count'].isna().sum()
distance_km_miss = data['distance_km'].isna().sum()
### Your code is here

In [ ]:
passanger_count_miss

In [ ]:
distance_km_miss

In [ ]:
# data.isna().sum()

### Число пассажиров

Заполните пропуски passenger_count наиболее частым непустым значением. Закомментированная строка решения ниже оставлена как незавершённая.

In [ ]:
### Your code is here
popular_value = data['passenger_count'].value_counts().index[0]
popular_value
# data['passenger_count'] = data['passenger_count'].fillna(popular_value)
# data['passenger_count'].unique()

### Расстояние

Заполните пропуски distance_km средним внутри групп по vendor_id, store_and_fwd_flag, traffic_jam, month_2, month_3, month_4, month_5, month_6. Это заданное приближение поиска похожих объектов.

In [ ]:

print("Количество пропусков в первых 20 строках:", data['distance_km'].head(20).isna().sum())



mean_8_features = data.groupby(['vendor_id','store_and_fwd_flag', 'traffic_jam','month_2','month_3','month_4','month_5','month_6'])['distance_km'].transform("mean")
data['distance_km'] = data['distance_km'].fillna(mean_8_features)


mean_soft = data.groupby(['vendor_id', 'traffic_jam'])['distance_km'].transform("mean")
data['distance_km'] = data['distance_km'].fillna(mean_soft)


data['distance_km'] = data['distance_km'].fillna(data['distance_km'].mean())


df_short = data.head(20)
df_short.to_csv('new_data3.csv', index=False, sep=';')

In [1]:
### Убедимся, что пропусков не осталось

### Your code is here

### Оценка после заполнения

Оцените линейную регрессию на кросс-валидации после обработки пропусков.

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

selector = KFold(n_splits=4, shuffle=True, random_state=33)
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression
model =  LinearRegression()


data = data.dropna(subset=['log_trip_duration'])
data = data.fillna(data.median(numeric_only=True))

X = data.drop('log_trip_duration', axis=1)
y = data['log_trip_duration']

scores = cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=selector)
cross_val_error = -scores.mean()

print(f"MSLE на Кросс-валидации: {round(cross_val_error, 3)}")

### Выбросы таргета

Постройте boxplot и гистограмму таргета. В исходной работе реализован только boxplot.

In [ ]:
import matplotlib.pyplot as plt
import  seaborn as sns
### Your code is here
fig = plt.figure()
fig.set_size_inches(10, 5)

# sns.barplot(data=data)
plt.boxplot(data['log_trip_duration'].values)
plt.ylim(0, 10)

plt.show()

### Фильтрация

Исследуйте выбросы и оставьте наблюдения между квантилями 0.005 и 0.995 таргета.

### Повторная оценка

После удаления выбросов повторите оценку с тем же selector.

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score


q_low = data['log_trip_duration'].quantile(0.005)
q_high = data['log_trip_duration'].quantile(0.995)

data_cleaned = data[(data['log_trip_duration'] > q_low) & (data['log_trip_duration'] < q_high)]


X_cleaned = data_cleaned.drop('log_trip_duration', axis=1)
y_cleaned = data_cleaned['log_trip_duration']


scores = cross_val_score(model, X_cleaned, y_cleaned, scoring='neg_mean_squared_error', cv=selector)


cross_val_error = -scores.mean()

print(f"MSLE на Кросс-валидации после чистки выбросов: {round(cross_val_error, 3)}")